# Install packages

# Import

In [1]:
import os
import time
import torch
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import QM9, ZINC
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Model

In [18]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class Regress_graph(torch.nn.Module):
    def __init__(self, num_layer, num_feature, num_hidden):
        super(Regress_graph, self).__init__()
        self.num_layers = num_layer
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_feature, num_hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(num_hidden, num_hidden))
        self.lt1 = torch.nn.Linear(num_hidden, 1)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, gc):
        x, edge_index, batch = gc.x, gc.edge_index, gc.batch
        x = x.float()
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = global_mean_pool(x, batch)
        x = self.lt1(x)
        return x

# Utils

In [11]:
def train_test_val_split(dataset, shuffle=True):
    N = len(dataset)
    if shuffle:
        idx = torch.randperm(N)
    else:
        idx = torch.arange(N)
    train = []
    val = []
    test = []
    for i in range(N):
        if i < N//2:
            train.append(dataset[idx[i]])
        elif i < 3*N//4 and i >= N//2:
            val.append(dataset[idx[i]])
        else:
            test.append(dataset[idx[i]])
    return train, test, val

In [27]:
def train_model(train_loader, model, loss_fn, optimizer, property=None):
  all_output_train = torch.tensor([]).to(device)
  all_labels_train = torch.tensor([]).to(device)
  train_loss = 0
  model.train()
  optimizer.zero_grad()

  for graphs in train_loader:
    graphs = graphs.to(device)
    out = model(graphs) #.astype(float)
    # print(out.shape, graphs.y.shape) # graphs.y[:, 0].view(-1, 1).shape
    loss = loss_fn(out, graphs.y.view(-1, 1).to(device))
    train_loss += loss.item()
    all_output_train = torch.cat((all_output_train, out))
    if property is not None:
        all_labels_train = torch.cat((all_labels_train, graphs.y[:, property].view(-1, 1).to(device)))
    else:
        all_labels_train = torch.cat((all_labels_train, graphs.y.to(device)))
    loss.backward()
    optimizer.step()

  train_loss = train_loss / len(train_loader)

  return train_loss / torch.std(all_labels_train).item()

def infer_model(loader, model, loss_fn, property=None):
  all_output = torch.tensor([]).to(device)
  all_labels = torch.tensor([]).to(device)
  all_loss = 0
  model.eval()

  for graphs in loader:
    graphs = graphs.to(device)
    out = model(graphs)
    loss = loss_fn(out, graphs.y.view(-1, 1).to(device))
    all_loss += loss.item()
    all_output = torch.cat((all_output, out))
    if property is not None:
        all_labels = torch.cat((all_labels, graphs.y[:, property].view(-1, 1).to(device)))
    else:
        all_labels = torch.cat((all_labels, graphs.y.to(device)))

  all_loss = all_loss / len(loader)

  return all_loss  / torch.std(all_labels).item()

# Main

In [28]:
dataset = ZINC(root='../dataset/ZINC', subset=True)
train_split, test_split, val_split = train_test_val_split(dataset, shuffle=True)
train_loader = DataLoader(train_split, batch_size=128, shuffle=True)
val_loader = DataLoader(val_split, batch_size=128, shuffle=False)
test_loader = DataLoader(test_split, batch_size=128, shuffle=False)

num_layer = 2
num_feature = dataset[0].x.shape[1]
num_hidden = 512
# property = 6
model = Regress_graph(num_layer, num_feature, num_hidden).to(device)
loss_fn = torch.nn.L1Loss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [29]:
if not os.path.exists("../save"):
  os.mkdir("../save")
if not os.path.exists("../save/graph_reg"):
  os.mkdir("../save/graph_reg")
if not os.path.exists("../save/graph_reg/baselines"):
  os.mkdir("../save/graph_reg/baselines")

best_val_loss = float('inf')
best_test_loss = float('inf')
best_val_acc = 0
best_test_acc = 0
all_train_loss = []
all_val_loss = []
all_test_loss = []
for epoch in tqdm(range(1)):
  #Train model
  train_loss = train_model(train_loader, model, loss_fn, optimizer)
  all_train_loss.append(train_loss)
  #Validate Model
  val_loss = infer_model(val_loader, model, loss_fn)
  all_val_loss.append(val_loss)
  #Test Model
  test_loss = infer_model(test_loader, model, loss_fn)
  all_test_loss.append(test_loss)
  #save model
  if val_loss <= best_val_loss or epoch == 0:
    best_val_loss = val_loss
    best_test_loss = test_loss
    torch.save(model.state_dict(), '../save/graph_reg/baselines/baseline_ZINC_subset.pt')
    print("\n")
    print(f"train loss: {train_loss}")
    print(f"val loss: {val_loss}")
    print(f"test loss: {test_loss}")
    print("Best model saved")
    print("\n")

  if epoch == 0 or epoch%25 == 0:
    print("\n")
    print(f"train loss: {train_loss}")
    print(f"val loss: {val_loss}")
    print(f"test loss: {test_loss}")
    print("\n")


print("\n")
print(f"Best Val Loss: {best_val_loss}")
print(f"Best Test Loss: {best_test_loss}")

100%|██████████| 1/1 [00:01<00:00,  1.08s/it]



train loss: 9.438037922022328
val loss: 32.59068133147795
test loss: 34.37760274054492
Best model saved




train loss: 9.438037922022328
val loss: 32.59068133147795
test loss: 34.37760274054492




Best Val Loss: 32.59068133147795
Best Test Loss: 34.37760274054492
